In [42]:
import pickle
import numpy as np

# Load the trained model and vectorizer
with open("disease_prediction_model.pkl", "rb") as model_file:
    model = pickle.load(model_file)

with open("tfidf_vectorizer.pkl", "rb") as vectorizer_file:
    vectorizer = pickle.load(vectorizer_file)

# List of disease names in the same order as the class indices
disease_labels = [
    "Calculus",
    "Caries",
    "Gingivitis",
    "Hypodontia",
    "Mouth Ulcer",
    "Oral Cancer",
    "Tooth Discoloration",
]

# Function for predicting disease with confidence scores
def predict_disease_with_scores(symptom_text):
    # Transform the input text using the vectorizer
    input_vector = vectorizer.transform([symptom_text])
    
    # Predict probabilities for all classes
    confidence_scores = model.predict_proba(input_vector)[0]
    
    # Combine disease labels and confidence scores
    disease_scores = dict(zip(disease_labels, confidence_scores))
    
    # Determine the predicted disease
    predicted_disease = max(disease_scores, key=disease_scores.get)
    
    return predicted_disease, disease_scores

# Test the prediction function
test_symptom = "red bleeding and swollen gums"
predicted_disease, scores = predict_disease_with_scores(test_symptom)

# Output results
print(f"Predicted disease for '{test_symptom}': {predicted_disease}")
print("Confidence scores for all classes:")
for disease, score in scores.items():
    print(f"{disease}: {score:.4f}")


Predicted disease for 'red bleeding and swollen gums': Gingivitis
Confidence scores for all classes:
Calculus: 0.0705
Caries: 0.0259
Gingivitis: 0.7382
Hypodontia: 0.0270
Mouth Ulcer: 0.0323
Oral Cancer: 0.0591
Tooth Discoloration: 0.0470


In [1]:
import tensorflow as tf
from tensorflow.keras.preprocessing import image
import numpy as np

# Define the list of disease labels (same order as used during training)
disease_labels = [
    "Oral_cancer", "Calculus", "Caries", "Gingivitis", "Hypodontia", "Mouth_Ulcer", "Tooth Discoloration"
]
# Load the trained CNN model (assuming it's in .h5 format)
model = tf.keras.models.load_model('mobilenet_model.h5')

# Function to preprocess and predict the disease from the uploaded image
def predict_disease_from_image(image_path):
    # Load and preprocess the image
    img = image.load_img(image_path, target_size=(128, 128))  # Adjusted size to match the model's expected input size
    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)  # Add batch dimension
    img_array /= 255.0  # Normalize the image (if required based on the model training)

    # Predict the disease
    predictions = model.predict(img_array)
    
    # Get the confidence scores for all classes
    confidence_scores = predictions[0]

    # Map the confidence scores to their respective class labels
    confidence_dict = {disease_labels[i]: confidence_scores[i] for i in range(len(disease_labels))}

    # Get the predicted class with the highest probability
    predicted_class_index = np.argmax(confidence_scores)
    predicted_disease = disease_labels[predicted_class_index]

    return predicted_disease, confidence_dict

# Example usage: replace with the path of your input image
image_path = "test/hypodontia/hypodontia_93.jpg"
predicted_disease, confidence_dict = predict_disease_from_image(image_path)

# Output the results
print(f"Predicted Disease: {predicted_disease}")
print("Confidence Scores:")
for disease, confidence in confidence_dict.items():
    print(f"{disease}: {confidence:.4f}")


ModuleNotFoundError: No module named 'tensorflow'

In [10]:
import numpy as np
import pickle
import tensorflow as tf
from tensorflow.keras.preprocessing import image

# Load models and vectorizer
with open("disease_prediction_model.pkl", "rb") as model_file:
    model = pickle.load(model_file)

with open("tfidf_vectorizer.pkl", "rb") as vectorizer_file:
    vectorizer = pickle.load(vectorizer_file)

image_model = tf.keras.models.load_model('mobilenet_model.h5')  # Image model

# List of disease names for the Text model
text_disease_labels = [
    "Calculus", "Caries", "Gingivitis", "Hypodontia", "Mouth_Ulcer", "Oral_cancer", "Tooth Discoloration"
]

# List of disease names for the Image model
image_disease_labels = [
    "Oral_cancer", "Calculus", "Caries", "Gingivitis", "Hypodontia", "Mouth_Ulcer", "Tooth Discoloration"
]

def debug_predictions(image_scores, text_scores):
    print("Image Model Confidence Scores:")
    for idx, score in enumerate(image_scores):
        print(f"{image_disease_labels[idx]}: {score:.4f}")
    
    print("\nText Model Confidence Scores:")
    for idx, score in enumerate(text_scores):
        print(f"{text_disease_labels[idx]}: {score:.4f}")

# Function to predict disease with confidence scores from the image model
def predict_image(image_path):
    img = image.load_img(image_path, target_size=(128, 128))  # Adjust size as per model input
    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)  # Add batch dimension
    img_array /= 255.0  # Normalize the image
    
    predictions = image_model.predict(img_array)
    return predictions[0]  # Return the class probabilities from the image model

# Function to predict disease with confidence scores from the text model
def predict_text(symptom_text):
    input_vector = vectorizer.transform([symptom_text])
    confidence_scores = model.predict_proba(input_vector)[0]
    return confidence_scores  # Return the class probabilities from the text model

# Function to process and debug the fusion logic
def decision_level_fusion(image_scores, text_scores):
    # Debugging: print scores of both models
    debug_predictions(image_scores, text_scores)

    # Get top 3 confidence scores for both image and text models
    image_sorted = sorted(enumerate(image_scores), key=lambda x: x[1], reverse=True)[:3]
    text_sorted = sorted(enumerate(text_scores), key=lambda x: x[1], reverse=True)[:3]

    # Apply 25% difference rule (optional)
    image_sorted = filter_25_percent_rule(image_sorted)
    text_sorted = filter_25_percent_rule(text_sorted)

    # Apply priority rules
    image_priority = assign_priority(image_sorted)
    text_priority = assign_priority(text_sorted)

    # Compare and decide final output based on highest combined confidence
    final_output = combine_confidence_scores(image_sorted, text_sorted)

    return final_output

# Function to filter out the least confidence score if the difference exceeds 25%
def filter_25_percent_rule(sorted_scores):
    if len(sorted_scores) > 1:
        top1_score = sorted_scores[0][1]
        top2_score = sorted_scores[1][1]
        top3_score = sorted_scores[2][1]
        
        if top1_score - top2_score > 0.25 * top1_score:
            sorted_scores = sorted_scores[:2]  # Remove the third one
        if top2_score - top3_score > 0.25 * top2_score:
            sorted_scores = sorted_scores[:2]  # Remove the third one
    return sorted_scores

# Function to assign priority based on confidence thresholds
def assign_priority(sorted_scores):
    priority = []
    for class_index, score in sorted_scores:
        if score >= 0.7:
            priority.append((class_index, score, "Primary"))
        elif score >= 0.5:
            priority.append((class_index, score, "Secondary"))
        else:
            priority.append((class_index, score, "Least Priority"))
    return priority

# Function to combine confidence scores and decide final output
def combine_confidence_scores(image_sorted, text_sorted):
    # Initialize the combined confidence scores for each class
    combined_confidences = {}

    # Combine the scores from both models for each class
    for idx, score in image_sorted:
        class_label = image_disease_labels[idx]
        combined_confidences[class_label] = combined_confidences.get(class_label, 0) + score

    for idx, score in text_sorted:
        class_label = text_disease_labels[idx]
        combined_confidences[class_label] = combined_confidences.get(class_label, 0) + score

    # Find the class with the highest combined confidence
    final_class = max(combined_confidences, key=combined_confidences.get)
    final_confidence = combined_confidences[final_class]

    print(f"Final Combined Confidence Scores: {combined_confidences}")
    print(f"\nFinal Predicted Disease: {final_class}")
    print(f"Confidence Score: {final_confidence:.4f}")

    return (final_class, final_confidence)

# Example usage: 
image_path = "v_mu.jpg"  # Image file path
text_input = " white painful sores which swell at times, pain when contact with hot or spicy food"  # Example text input

# Predict from both models
image_scores = predict_image(image_path)
text_scores = predict_text(text_input)

# Apply decision-level fusion
final_output = decision_level_fusion(image_scores, text_scores)

# Output the final decision
print(f"\nFinal Predicted Disease: {final_output[0]}")
print(f"Confidence Score: {final_output[1]:.4f}")


ImportError: Could not import PIL.Image. The use of `load_img` requires PIL.

In [47]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import joblib  # To save the trained model

# Load the dataset
df = pd.read_csv("final_dataset.csv")

# Define the feature columns (confidence scores from image and text models)
image_features = [col for col in df.columns if col.startswith("image_conf")]
text_features = [col for col in df.columns if col.startswith("text_conf")]
features = image_features + text_features  # Combine image and text features

# Target variable (true label)
target = 'true_label'

# Split the data into features (X) and target (y)
X = df[features]  # Feature columns
y = df[target]    # True labels

# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train the Logistic Regression model
log_reg_model = LogisticRegression(max_iter=1000, random_state=42)  # Increase max_iter if convergence is an issue
log_reg_model.fit(X_train, y_train)

# Evaluate the model on the test set
y_pred = log_reg_model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Logistic Regression model accuracy: {accuracy:.4f}")

# Save the trained model to a file
joblib.dump(log_reg_model, 'final_trained_log_reg_model.pkl')
print("Model saved as 'final_trained_log_reg_model.pkl'")


Logistic Regression model accuracy: 0.9107
Model saved as 'final_trained_log_reg_model.pkl'


In [29]:
import numpy as np
import pickle
import tensorflow as tf
from tensorflow.keras.preprocessing import image
import joblib  # Correct import for joblib

# Load models and vectorizer
with open("disease_prediction_model.pkl", "rb") as model_file:
    model = pickle.load(model_file)

with open("tfidf_vectorizer.pkl", "rb") as vectorizer_file:
    vectorizer = pickle.load(vectorizer_file)

image_model = tf.keras.models.load_model('mobilenet_model.h5')  # Image model

# List of disease names for the Text model
text_disease_labels = [
    "Calculus", "Caries", "Gingivitis", "Hypodontia", "Mouth_Ulcer", "Oral_cancer", "Tooth Discoloration"
]

# List of disease names for the Image model
image_disease_labels = [
    "Oral_cancer", "Calculus", "Caries", "Gingivitis", "Hypodontia", "Mouth_Ulcer", "Tooth Discoloration"
]

# Function to predict disease with confidence scores from the image model
def predict_image(image_path):
    img = image.load_img(image_path, target_size=(128, 128))  # Adjust size as per model input
    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)  # Add batch dimension
    img_array /= 255.0  # Normalize the image
    
    predictions = image_model.predict(img_array)
    return predictions[0]  # Return the class probabilities from the image model

# Function to predict disease with confidence scores from the text model
def predict_text(symptom_text):
    input_vector = vectorizer.transform([symptom_text])
    confidence_scores = model.predict_proba(input_vector)[0]
    return confidence_scores  # Return the class probabilities from the text model

# Function to load the trained Logistic Regression model
def load_logistic_regression_model():
    return joblib.load("final_trained_log_reg_model.pkl")

# Function to combine the image and text model confidence scores for final prediction using Logistic Regression
def predict_final_disease(image_scores, text_scores):
    # Ensure both image_scores and text_scores are of length 7
    if len(image_scores) != 7 or len(text_scores) != 7:
        raise ValueError("Both image_scores and text_scores must contain 7 values each.")
    
    # Reorder image scores based on the text model labels (final_dataset requires text_disease_labels order)
    ordered_image_scores = [0] * len(text_disease_labels)
    for idx, disease in enumerate(image_disease_labels):
        if disease in text_disease_labels:
            ordered_index = text_disease_labels.index(disease)
            ordered_image_scores[ordered_index] = image_scores[idx]

    # Combine image and text model confidence scores as features for Logistic Regression
    combined_features = np.concatenate([ordered_image_scores, text_scores])  # This should have 14 features (7 image + 7 text)
    print("Combined Features Shape:", combined_features.shape)

    
    # Ensure the combined features are in the correct format (no feature names)
    log_reg_model = load_logistic_regression_model()
    
    # Predict the disease using the Logistic Regression model
    final_prediction = log_reg_model.predict(combined_features.reshape(1, -1))  # Reshape for one sample
    final_confidence = log_reg_model.predict_proba(combined_features.reshape(1, -1))[0]  # Get all probabilities for the 7 classes
    
    # Map the predicted class and confidence scores with the disease labels
    label_confidence_dict = {text_disease_labels[i]: final_confidence[i] for i in range(len(text_disease_labels))}
    
    # Sort the dictionary by confidence score in descending order
    sorted_labels = sorted(label_confidence_dict.items(), key=lambda x: x[1], reverse=True)

    print(f"Confidence Scores with Labels:")
    for label, confidence in sorted_labels:
        print(f"{label}: {confidence:.4f}")
    
    # Print Image Scores with corresponding labels
    print("\nImage Scores with Labels:")
    for idx, score in enumerate(ordered_image_scores):
        print(f"{text_disease_labels[idx]}: {score:.4f}")
    
    # Print Text Scores with corresponding labels
    print("\nText Scores with Labels:")
    for idx, score in enumerate(text_scores):
        print(f"{text_disease_labels[idx]}: {score:.4f}")
    
    final_class = final_prediction[0]
    print(f"\nFinal Predicted Disease: {final_class}")
    print(f"Confidence Score: {final_confidence[text_disease_labels.index(final_class)]:.4f}")
    
    return final_class, final_confidence

# Example usage: 
image_path = "test/caries/caries_54.jpeg"  # Image file path
text_input = "My teeth has turned to black and i get bad smell from my mouth"  # Example text input

# Predict from both models
image_scores = predict_image(image_path)
text_scores = predict_text(text_input)

# Ensure both image_scores and text_scores are of length 7
print(f"Image Scores: {image_scores}")
print(f"Text Scores: {text_scores}")

# Check if the scores' lengths are 7 each before proceeding
if len(image_scores) == 7 and len(text_scores) == 7:
    # Predict final disease using Logistic Regression model
    final_output = predict_final_disease(image_scores, text_scores)

    # Output the final decision
    print(f"\nFinal Predicted Disease: {final_output[0]}")
    print(f"Confidence Score: {final_output[1][text_disease_labels.index(final_output[0])]:.4f}")
else:
    print("Error: The number of features in image_scores or text_scores is not correct.")


C:\Users\abhij\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:380: InconsistentVersionWarning: Trying to unpickle estimator LogisticRegression from version 1.5.1 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
C:\Users\abhij\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:380: InconsistentVersionWarning: Trying to unpickle estimator TfidfTransformer from version 1.5.1 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
C:\Users\abhij\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:380: InconsistentVersionWarning: Trying to unpickle e

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
Image Scores: [0.00204024 0.04597782 0.67939043 0.00679599 0.08438886 0.00385992
 0.17754678]
Text Scores: [0.12702068 0.05763395 0.09203927 0.11344704 0.05673046 0.08619073
 0.46693787]
Combined Features Shape: (14,)
Confidence Scores with Labels:
Caries: 0.6733
Tooth Discoloration: 0.2794
Hypodontia: 0.0120
Calculus: 0.0108
Oral_cancer: 0.0106
Mouth_Ulcer: 0.0079
Gingivitis: 0.0061

Image Scores with Labels:
Calculus: 0.0460
Caries: 0.6794
Gingivitis: 0.0068
Hypodontia: 0.0844
Mouth_Ulcer: 0.0039
Oral_cancer: 0.0020
Tooth Discoloration: 0.1775

Text Scores with Labels:
Calculus: 0.1270
Caries: 0.0576
Gingivitis: 0.0920
Hypodontia: 0.1134
Mouth_Ulcer: 0.0567
Oral_cancer: 0.0862
Tooth Discoloration: 0.4669

Final Predicted Disease: Caries
Confidence Score: 0.6733

Final Predicted Disease: Caries
Confidence Score: 0.6733


C:\Users\abhij\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(
C:\Users\abhij\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(


In [7]:
!pip install Pillow

   ---------------------------------------- 0.0/2.6 MB ? eta -:--:--
   --- ------------------------------------ 0.3/2.6 MB ? eta -:--:--
   --------------- ------------------------ 1.0/2.6 MB 3.6 MB/s eta 0:00:01
   --------------------------- ------------ 1.8/2.6 MB 3.6 MB/s eta 0:00:01
   ---------------------------------------- 2.6/2.6 MB 3.6 MB/s eta 0:00:00



[notice] A new release of pip is available: 24.2 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [13]:
import sys
print(sys.executable)

C:\Users\abhij\AppData\Local\Programs\Python\Python313\python.exe
